In [1]:
import nibabel as nb
import numpy as np

In [2]:
import matplotlib.pyplot as plt

In [3]:
dyads1 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/dyads1.nii.gz")
dyads2 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/dyads2.nii.gz")
dyads3 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/dyads3.nii.gz")

dyads1_dispersion = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/dyads1_dispersion.nii.gz")
dyads2_dispersion = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/dyads2_dispersion.nii.gz")
dyads3_dispersion = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/dyads3_dispersion.nii.gz")

dyads1_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/reordered/dyads1.nii.gz")
dyads2_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/reordered/dyads2.nii.gz")
dyads3_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/reordered/dyads3.nii.gz")

f1_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/reordered/mean_f1samples.nii.gz")
f2_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/reordered/mean_f2samples.nii.gz")
f3_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/reordered/mean_f3samples.nii.gz")

dyads1_dispersion_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/reordered/dyads1_dispersion.nii.gz")
dyads2_dispersion_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/reordered/dyads2_dispersion.nii.gz")
dyads3_dispersion_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/reordered/dyads3_dispersion.nii.gz")



mean_f0 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/mean_f0samples.nii.gz")
mean_f1 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/mean_f1samples.nii.gz")
mean_f2 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/mean_f2samples.nii.gz")
mean_f3 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/mean_f3samples.nii.gz")

f0 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/merged_f0samples.nii.gz")
f1 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/merged_f1samples.nii.gz")
f2 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/merged_f2samples.nii.gz")
f3 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_no_selection/merged_f3samples.nii.gz")


In [4]:
# Fractions MCCM

f0_mcmc = nb.load("data/MCMC_ARD_3fib/mean_f1samples.nii.gz").get_fdata()
f1_mcmc = nb.load("data/MCMC_ARD_3fib/mean_f2samples.nii.gz").get_fdata()
f2_mcmc = nb.load("data/MCMC_ARD_3fib/mean_f3samples.nii.gz").get_fdata()

fsum_mcmc = f0_mcmc + f1_mcmc + f2_mcmc


dyads_mcmc = nb.load("data/MCMC_ARD_3fib/dyads1.nii.gz").get_fdata()
dyads2_mcmc = nb.load("data/MCMC_ARD_3fib/dyads2.nii.gz").get_fdata()
dyads3_mcmc = nb.load("data/MCMC_ARD_3fib/dyads3.nii.gz").get_fdata()

dyads_mcmc = np.stack([dyads_mcmc, dyads2_mcmc, dyads3_mcmc], axis=-2)
fractions_mcmc = np.stack([f0_mcmc, f1_mcmc, f2_mcmc], axis=-1)


In [5]:
from scipy.ndimage import binary_erosion

struct = np.ones((3,3,3))
eroded_mask = binary_erosion(fsum_mcmc > 0, structure=struct, iterations=1)

fractions_mcmc = np.where(eroded_mask[...,None], fractions_mcmc, 0)

In [16]:
# Estiamted fraction

fractions_sbi = np.stack([f1.get_fdata(), f2.get_fdata(), f3.get_fdata()], axis=-1).mean(-2)
fractions_sbi = np.where(eroded_mask[...,None], fractions_sbi, 0)

dyads_sbi = np.stack([dyads1.get_fdata(), dyads2.get_fdata(), dyads3.get_fdata()], axis=-2)

In [ ]:
fractions_sbi.shape

In [8]:
from dmri.utils.viz import orthoview_ultracompact, save_orthoview_html

fig = orthoview_ultracompact(fractions_mcmc, precision=np.float16, vmin=0, vmax=0.8, heatmap_quality=0.8, channel_names=["f1", "f2", "f3"])
save_orthoview_html(fig, "fractions_mcmc.html")


In [9]:
fig = orthoview_ultracompact(fractions_sbi, precision=np.float16, vmin=0, vmax=0.8, heatmap_quality=0.8, channel_names=["f1", "f2", "f3"])
save_orthoview_html(fig, "fractions_sbi.html")

In [10]:
# Differene between estimated fractions

fractions_diff = fractions_mcmc - fractions_sbi

fig = orthoview_ultracompact(fractions_diff, precision=np.float16, vmin=-0.2, vmax=0.2, heatmap_quality=0.8,
                            color_map=[[0, 'red'], [0.5, 'black'], [1, 'green']],
                            channel_names=["f1", "f2", "f3"])
save_orthoview_html(fig, "fractions_diff.html")


In [11]:
#fig

In [12]:
from dmri.utils.viz import save_orthoview_html, orthoview_quiver_ultracompact

In [22]:
fig = orthoview_quiver_ultracompact(dyads_mcmc,fractions_mcmc, show_all_channels=True, precision=np.float16, heatmap_quality=0.8, arrow_width=1.5)
save_orthoview_html(fig, "dyads3d_mcmc.html")

In [23]:
fig = orthoview_quiver_ultracompact(dyads_sbi,fractions_sbi, show_all_channels=True, precision=np.float16, heatmap_quality=0.8, arrow_width=1.5)
save_orthoview_html(fig, "dyads3d_sbi.html")

In [ ]:
fig.show()

In [22]:
fig.write_html("dyads3d_mcmc.html")

In [ ]:
fig.show()

In [189]:
fig_fraction_mcmc = orthoview_plotly(fractions_mcmc)

In [191]:
fig_dyads_mcmc = orthoview_quiver_plotly(dyads_mcmc,fractions_mcmc)
fig_dyads_mcmc.write_html("dyads_mcmc.html")


In [ ]:
fractions_diff = fractions_mcmc - fractions_mcmc.mean(-1, keepdims=True)
fig_fraction_mcmc = orthoview_plotly(fractions_diff)

In [190]:
fig_fraction_mcmc.write_html("fractions_mcmc.html")

In [177]:
dyads = np.stack([dyads1.get_fdata(), dyads2.get_fdata(), dyads3.get_fdata()], axis=-2)

In [178]:
fractions = np.stack([mean_f1.get_fdata(), mean_f2.get_fdata(), mean_f3.get_fdata()], axis=-1)

In [184]:
fig = orthoview_quiver_plotly(dyads,fractions,step=1)

In [ ]:
fig.show()

In [185]:
fig.write_html("dyads.html")

In [4]:
brain_mask = nb.load("data/nodif_brain_mask.nii.gz").get_fdata().astype(bool)

In [ ]:
dyads1_dispersion.orthoview()

In [ ]:
dyads1_dispersion_reordered.orthoview()

In [52]:
# Create grid

# Reordered
dir_data = dyads1.get_fdata()
dir_data2 = dyads2.get_fdata()
dir_data3 = dyads3.get_fdata()
f1 = f1.get_fdata().mean(-1)
f2 = f2.get_fdata().mean(-1)
f3 = f3.get_fdata().mean(-1)
fsum = f1 + f2 + f3

f1 = f1 / np.where(fsum > 0, fsum, 1)
f2 = f2 / np.where(fsum > 0, fsum, 1)
f3 = f3 / np.where(fsum > 0, fsum, 1)

weighted_dir_data = f1[...,None]*dir_data
weighted_dir_data2 = f2[...,None]*dir_data2
weighted_dir_data3 = f3[...,None]*dir_data3
x, y = np.meshgrid(np.arange(dir_data.shape[0]), np.arange(dir_data.shape[1]), indexing='ij')

z_slice = 35
background_img = fsum[:,:,z_slice]

u = weighted_dir_data[:, :, z_slice, 0]
v = weighted_dir_data[:, :, z_slice, 1]

u2 = weighted_dir_data2[:, :, z_slice, 0]
v2 = weighted_dir_data2[:, :, z_slice, 1]

u3 = weighted_dir_data3[:, :, z_slice, 0]
v3 = weighted_dir_data3[:, :, z_slice, 1]


In [ ]:
%matplotlib inline
# Plot
plt.figure(figsize=(15, 15))
#plt.imshow(background_slice.T, cmap='gray', origin='lower')
plt.imshow(background_img.T, cmap='gray', origin='lower')
plt.quiver(x, y, u, v, color='red', alpha=f1[:,:,z_slice], scale=100, headwidth=0, headlength=0, headaxislength=0)
plt.quiver(x,y, -u, -v, color='red', alpha=f1[:,:,z_slice], scale=100, headwidth=0, headlength=0, headaxislength=0)
plt.quiver(x, y, u2, v2, color='blue', alpha=f2[:,:,z_slice], scale=100, headwidth=0, headlength=0, headaxislength=0)
plt.quiver(x,y, -u2, -v2, color='blue', alpha=f2[:,:,z_slice], scale=100, headwidth=0, headlength=0, headaxislength=0)
plt.quiver(x, y, u3, v3, color='green', alpha=f3[:,:,z_slice], scale=100, headwidth=0, headlength=0, headaxislength=0)
plt.quiver(x,y, -u3, -v3, color='green', alpha=f3[:,:,z_slice], scale=100, headwidth=0, headlength=0, headaxislength=0)
plt.title(f'Diffusion directions at slice z={z_slice}')
plt.axis('off')
plt.show()

In [13]:
dyads1 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/dyads1.nii.gz")
dyads2 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/dyads2.nii.gz")
dyads3 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/dyads3.nii.gz")

dyads1_dispersion = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/dyads1_dispersion.nii.gz")
dyads2_dispersion = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/dyads2_dispersion.nii.gz")
dyads3_dispersion = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/dyads3_dispersion.nii.gz")

dyads1_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/reordered/dyads1.nii.gz")
dyads2_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/reordered/dyads2.nii.gz")
dyads3_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/reordered/dyads3.nii.gz")

f1_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/reordered/merged_f1samples.nii.gz")
f2_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/reordered/merged_f2samples.nii.gz")
f3_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/reordered/merged_f3samples.nii.gz")

dyads1_dispersion_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/reordered/dyads1_dispersion.nii.gz")
dyads2_dispersion_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/reordered/dyads2_dispersion.nii.gz")
dyads3_dispersion_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/reordered/dyads3_dispersion.nii.gz")



mean_f0 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/mean_f0samples.nii.gz")
mean_f1 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/mean_f1samples.nii.gz")
mean_f2 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/mean_f2samples.nii.gz")
mean_f3 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/mean_f3samples.nii.gz")

_f0 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/merged_f0samples.nii.gz")
_f1 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/merged_f1samples.nii.gz")
_f2 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/merged_f2samples.nii.gz")
_f3 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_best_model_selection/merged_f3samples.nii.gz")


In [7]:
dyads1 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/dyads1.nii.gz")
dyads2 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/dyads2.nii.gz")
dyads3 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/dyads3.nii.gz")

dyads1_dispersion = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/dyads1_dispersion.nii.gz")
dyads2_dispersion = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/dyads2_dispersion.nii.gz")
dyads3_dispersion = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/dyads3_dispersion.nii.gz")

dyads1_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/reordered/dyads1.nii.gz")
dyads2_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/reordered/dyads2.nii.gz")
dyads3_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/reordered/dyads3.nii.gz")

f1_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/reordered/merged_f1samples.nii.gz")
f2_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/reordered/merged_f2samples.nii.gz")
f3_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/reordered/merged_f3samples.nii.gz")

dyads1_dispersion_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/reordered/dyads1_dispersion.nii.gz")
dyads2_dispersion_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/reordered/dyads2_dispersion.nii.gz")
dyads3_dispersion_reordered = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/reordered/dyads3_dispersion.nii.gz")



mean_f0 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/mean_f0samples.nii.gz")
mean_f1 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/mean_f1samples.nii.gz")
mean_f2 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/mean_f2samples.nii.gz")
mean_f3 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/mean_f3samples.nii.gz")

_f0 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/merged_f0samples.nii.gz")
_f1 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/merged_f1samples.nii.gz")
_f2 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/merged_f2samples.nii.gz")
_f3 = nb.load("../results/full_ball3stick_simformer6/ball3stick_inference_results_average_model_selection/merged_f3samples.nii.gz")


In [8]:
# Create grid

# Reordered
dir_data = dyads1.get_fdata()
dir_data2 = dyads2.get_fdata()
dir_data3 = dyads3.get_fdata()
fsum = np.mean(_f1.get_fdata() + _f2.get_fdata()+ _f3.get_fdata(), axis=-1)
f1 = _f1.get_fdata().mean(-1)
f2 = _f2.get_fdata().mean(-1)
f3 = _f3.get_fdata().mean(-1)


f1 = f1 / np.where(fsum > 0, fsum, 1)
f2 = f2 / np.where(fsum > 0, fsum, 1)
f3 = f3 / np.where(fsum > 0, fsum, 1)

weighted_dir_data = f1[...,None]*dir_data
weighted_dir_data2 = f2[...,None]*dir_data2
weighted_dir_data3 = f3[...,None]*dir_data3
x, y = np.meshgrid(np.arange(dir_data.shape[0]), np.arange(dir_data.shape[1]), indexing='ij')

z_slice = 35
background_img = fsum[:,:,z_slice]

u = weighted_dir_data[:, :, z_slice, 0]
v = weighted_dir_data[:, :, z_slice, 1]

u2 = weighted_dir_data2[:, :, z_slice, 0]
v2 = weighted_dir_data2[:, :, z_slice, 1]

u3 = weighted_dir_data3[:, :, z_slice, 0]
v3 = weighted_dir_data3[:, :, z_slice, 1]


In [ ]:
%matplotlib inline
# Plot
plt.figure(figsize=(15, 15))
#plt.imshow(background_slice.T, cmap='gray', origin='lower')
plt.imshow(background_img.T, cmap='gray', origin='lower')
plt.quiver(x, y, u, v, color='red', alpha=f1[:,:,z_slice], scale=100, headwidth=0, headlength=0, headaxislength=0)
plt.quiver(x,y, -u, -v, color='red', alpha=f1[:,:,z_slice], scale=100, headwidth=0, headlength=0, headaxislength=0)
plt.quiver(x, y, u2, v2, color='blue', alpha=f2[:,:,z_slice], scale=100, headwidth=0, headlength=0, headaxislength=0)
plt.quiver(x,y, -u2, -v2, color='blue', alpha=f2[:,:,z_slice], scale=100, headwidth=0, headlength=0, headaxislength=0)
plt.quiver(x, y, u3, v3, color='green', alpha=f3[:,:,z_slice], scale=100, headwidth=0, headlength=0, headaxislength=0)
plt.quiver(x,y, -u3, -v3, color='green', alpha=f3[:,:,z_slice], scale=100, headwidth=0, headlength=0, headaxislength=0)
plt.title(f'Diffusion directions at slice z={z_slice}')
plt.axis('off')
plt.show()

In [10]:
# Erode the brain mask to remove border artifacts
from scipy.ndimage import binary_erosion

# Create a more conservative brain mask by eroding the original one
# The structure parameter determines how aggressive the erosion is
# Using a 3x3x3 cubic structure element
struct = np.ones((3,3,3))
eroded_mask = binary_erosion(fsum > 0, structure=struct, iterations=2)

# Apply the eroded mask to fsum
fsum_cleaned = fsum * eroded_mask


In [ ]:
import plotly.graph_objects as go
import numpy as np
from scipy.ndimage import zoom

# Your data
# fsum = <your 104x104x72 data>

# Downsample by 2x in all axes
fsum_downsampled = zoom(fsum_cleaned, zoom=(0.9, 0.9, 0.9), order=1)  # linear interpolation

# Create coordinates
x, y, z = np.meshgrid(np.arange(fsum_downsampled.shape[0]),
                      np.arange(fsum_downsampled.shape[1]),
                      np.arange(fsum_downsampled.shape[2]),
                      indexing='ij')

# Calculate opacity based on fsum values
# Scale fsum values to 0-1 range for opacity
min_val = 0.
max_val = 0.9

fig = go.Figure(data=go.Volume(
    x=x.flatten(),
    y=y.flatten(),
    z=z.flatten(),
    value=fsum_downsampled.flatten(),
    isomin=min_val,
    isomax=max_val,
    opacity=0.2,
    opacityscale=[
        [0.0, 0.0],  # At min value → fully transparent
        [0.2, 0.0],  # Still transparent up to 20%
        [0.4, 0.1],  # Start becoming slightly visible
        [0.5, 0.3],  # Start becoming slightly visible
        [0.7, 0.6],
        [1.0, 1.0],
    ],
    surface_count=15,
    colorscale='greys_r'
))

# Update layout to set black background and hide axes
fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor="black"          # Black background
    ),
    paper_bgcolor="black",       # Entire figure background black
    plot_bgcolor="black",
)
fig.write_html("fsum.html")
fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np
from scipy.ndimage import zoom

# Your data
# fsum = <your 104x104x72 data>

# Downsample by 2x in all axes
fsum_downsampled = zoom(fsum_cleaned, zoom=(0.5, 0.5, 0.5), order=1)  # linear interpolation

# Create coordinates
x, y, z = np.meshgrid(np.arange(fsum_downsampled.shape[0]),
                      np.arange(fsum_downsampled.shape[1]),
                      np.arange(fsum_downsampled.shape[2]),
                      indexing='ij')

# Calculate opacity based on fsum values
# Scale fsum values to 0-1 range for opacity
min_val = 0.
max_val = 0.9

fig = go.Figure(data=go.Volume(
    x=x.flatten(),
    y=y.flatten(),
    z=z.flatten(),
    value=fsum_downsampled.flatten(),
    isomin=min_val,
    isomax=max_val,
    opacity=1.,
    opacityscale=[
        [0.0, 0.0],  # At min value → fully transparent
        [0.2, 0.0],  # Still transparent up to 20%
        [0.4, 0.01],  # Start becoming slightly visible
        [0.5, 0.2],  # Start becoming slightly visible
        [0.7, 0.6],
        [1.0, 1.0],
    ],
    surface_count=20,
    colorscale='greys_r'
))

# Update layout to set black background and hide axes
fig.update_layout(
    scene=dict(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor="black"          # Black background
    ),
    paper_bgcolor="black",       # Entire figure background black
    plot_bgcolor="black",
)
fig.write_html("fsum_small.html")
fig.show()